In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score
plt.style.use('ggplot')
sns.set_palette('husl')
%matplotlib inline

# Càrrega de dades
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print()
print(train.head())

Train: (891, 12)
Test:  (418, 11)

   PassengerId  Survived  Pclass   
0            1         0       3  \
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp   
0                            Braund, Mr. Owen Harris    male  22.0      1  \
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            

In [2]:
def netejar_dades(df):
    df = df.copy()
    
    # PRIMER — capturar els nuls originals abans d'imputar
    df['EsImmigrantPobre'] = (
        (df['Age'].isna()) & 
        (df['Cabin'].isna()) & 
        (df['Pclass'] == 3)
    ).astype(int)
    
    # 1. Extreure títol
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(
        ['Lady','Countess','Capt','Col','Don',
         'Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare'
    )
    df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
    
    # 2. Features de família PRIMER (necessitem FamilySize per imputar edat)
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['TicketSize'] = df['Ticket'].map(df['Ticket'].value_counts())
    df['GroupSize']  = df[['FamilySize','TicketSize']].max(axis=1)
    df['GroupSize_Cat'] = pd.cut(
        df['GroupSize'],
        bins=[0, 1, 4, 100],
        labels=['sol', 'optim', 'gran']
    )
    
    # 3. Imputar edat per títol + classe
    mediana_edat = df.groupby(['Title','Pclass'])['Age'].median()
    for title in df['Title'].unique():
        for pclass in [1, 2, 3]:
            mask = (df['Title'] == title) & (df['Pclass'] == pclass) & (df['Age'].isna())
            if mask.sum() > 0:
                df.loc[mask, 'Age'] = mediana_edat.get((title, pclass), df['Age'].median())
    
    # Millora: Miss de 3a classe per mida família
    mask_miss = (df['Title'] == 'Miss') & (df['Pclass'] == 3) & (df['Age'].isna())
    df.loc[mask_miss & (df['FamilySize'] >= 3), 'Age'] = 8
    df.loc[mask_miss & (df['FamilySize'] < 3), 'Age']  = 18
    
    # 4. Imputar Embarked
    df['Embarked'] = df['Embarked'].fillna('S')
    
    # 5. Coberta
    df['Deck'] = df.apply(
        lambda x: x['Cabin'][0] if pd.notna(x['Cabin']) 
        else f"U{x['Pclass']}", 
        axis=1
    )
    df['Deck_Risk'] = df['Deck'].map({
        'A': 0, 'B': 0, 'C': 1,
        'D': 0, 'E': 1, 'F': 0,
        'G': 0, 'U1': 0, 'U2': 0, 'U3': 0
    }).fillna(0)
    
    # 6. Interacció Sexe x Classe
    df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)
    
    # 7. Fare logarítmic
    df['Fare_log'] = np.log1p(df['Fare'])
    
    # 8. Banda d'edat
    df['Age_band'] = pd.cut(
        df['Age'],
        bins=[0, 12, 18, 35, 60, 100],
        labels=['nen','adolescent','jove','adult','gran']
    )
    
    # 9. Prefix ticket
    df['TicketPrefix'] = df['Ticket'].str.extract(r'^([A-Za-z./]+)', expand=False).fillna('NUM')
    df['TicketPrefix'] = df['TicketPrefix'].apply(
        lambda x: x if x in ['PC','NUM'] else 'OTHER'
    )
    
    # 10. Interaccions edat × sexe × classe
    df['Is_Child']  = (df['Age'] < 12).astype(int)
    df['Child_3rd'] = ((df['Age'] < 12) & (df['Pclass'] == 3)).astype(int)
    df['Woman_3rd'] = ((df['Sex'] == 'female') & (df['Pclass'] == 3)).astype(int)
    df['Man_1st']   = ((df['Sex'] == 'male') & (df['Pclass'] == 1)).astype(int)
    
    # 11. Costat del vaixell
    df['Cabin_Num'] = df['Cabin'].str.extract(r'(\d+)', expand=False).astype(float)
    df['Cabin_Side'] = df['Cabin_Num'].apply(
        lambda x: 'E' if pd.notna(x) and x % 2 == 0 
        else 'B' if pd.notna(x) 
        else 'U'
    )
    df['Side_Pclass'] = df['Cabin_Side'] + '_' + df['Pclass'].astype(str)
    
    # 12. Model causal 3 fases
    df['Porta_Tancada']  = ((df['Pclass'] == 3) & (df['FamilySize'] >= 5)).astype(int)
    df['Man_WithFamily'] = ((df['Sex'] == 'male') & (df['FamilySize'] > 1)).astype(int)
    df['Man_Solo_Low']   = ((df['Sex'] == 'male') & (df['FamilySize'] == 1) & (df['Pclass'] >= 2)).astype(int)
    
    # 13. Coberta C risc homes 1a
    df['Man1st_CobertaC'] = ((df['Sex'] == 'male') & (df['Pclass'] == 1) & (df['Deck'] == 'C')).astype(int)
    df['Man1st_SafeDeck'] = ((df['Sex'] == 'male') & (df['Pclass'] == 1) & (df['Deck'].isin(['D','E']))).astype(int)
    
    return df

# Aplicar
train_net = netejar_dades(train)
test_net  = netejar_dades(test)
print("Neteja completada!")
print(f"Features disponibles: {len(train_net.columns)}")

Neteja completada!
Features disponibles: 36


In [3]:
# Holdout set - separar ABANS de tot
train_real, holdout = train_test_split(
    train, test_size=0.2, random_state=42, stratify=train['Survived']
)

train_real_net = netejar_dades(train_real)
holdout_net    = netejar_dades(holdout)

print(f"Train real: {len(train_real)} passatgers")
print(f"Holdout:    {len(holdout)} passatgers")
print(f"Supervivència train: {train_real['Survived'].mean():.1%}")
print(f"Supervivència holdout: {holdout['Survived'].mean():.1%}")

# Features base V2 (millor Kaggle: 0.7823)
features_v2 = [
    'Sex_Pclass', 'Title', 'Fare_log',
    'Pclass', 'FamilySize', 'TicketSize'
]

# Funció d'avaluació amb holdout
def avaluar(features, nom, model=None):
    if model is None:
        model = RandomForestClassifier(
            n_estimators=300, max_depth=4,
            min_samples_leaf=5, random_state=42
        )
    
    df_tr = train_real_net.copy()
    df_ho = holdout_net.copy()
    
    cols_cat = [c for c in ['Sex_Pclass','Title','Age_band','TicketPrefix',
                             'Embarked','Deck','GroupSize_Cat','Cabin_Side','Side_Pclass'] 
                if c in features]
    
    for col in cols_cat:
        le = LabelEncoder()
        combined = pd.concat([df_tr[col], df_ho[col]]).astype(str)
        le.fit(combined)
        df_tr[col] = le.transform(df_tr[col].astype(str))
        df_ho[col] = le.transform(df_ho[col].astype(str))
    
    X_tr = df_tr[features]
    y_tr = df_tr['Survived']
    X_ho = df_ho[features]
    y_ho = df_ho['Survived']
    
    cv = cross_val_score(model, X_tr, y_tr, cv=5, scoring='accuracy').mean()
    model.fit(X_tr, y_tr)
    ho = accuracy_score(y_ho, model.predict(X_ho))
    
    print(f"{nom:45s} CV: {cv:.4f}  Holdout: {ho:.4f}")
    return model, cv, ho

# Avaluar base
avaluar(features_v2, 'Base V2')

Train real: 712 passatgers
Holdout:    179 passatgers
Supervivència train: 38.3%
Supervivència holdout: 38.5%
Base V2                                       CV: 0.8189  Holdout: 0.8101


(RandomForestClassifier(max_depth=4, min_samples_leaf=5, n_estimators=300,
                        random_state=42),
 0.8188909681867429,
 0.8100558659217877)

In [4]:
# Funció per generar submission
def generar_submission(features, nom_fitxer, model=None):
    if model is None:
        model = RandomForestClassifier(
            n_estimators=300, max_depth=4,
            min_samples_leaf=5, random_state=42
        )
    
    # Entrenar amb TOT el train (no només train_real)
    df_tr = train_net.copy()
    df_te = test_net.copy()
    
    cols_cat = [c for c in ['Sex_Pclass','Title','Age_band','TicketPrefix',
                             'Embarked','Deck','GroupSize_Cat','Cabin_Side','Side_Pclass'] 
                if c in features]
    
    for col in cols_cat:
        le = LabelEncoder()
        combined = pd.concat([df_tr[col], df_te[col]]).astype(str)
        le.fit(combined)
        df_tr[col] = le.transform(df_tr[col].astype(str))
        df_te[col] = le.transform(df_te[col].astype(str))
    
    # Imputar NaN
    df_te['Fare_log'] = df_te['Fare_log'].fillna(df_te['Fare_log'].median())
    
    model.fit(df_tr[features], df_tr['Survived'])
    predictions = model.predict(df_te[features])
    
    submission = pd.DataFrame({
        'PassengerId': test['PassengerId'],
        'Survived':    predictions
    })
    
    submission.to_csv(f'../submissions/{nom_fitxer}', index=False)
    print(f"Submission guardada: {nom_fitxer}")
    print(submission['Survived'].value_counts())
    return submission

print("Pipeline complet i llest!")

Pipeline complet i llest!
